In [ ]:
# -*- coding: utf-8 -*-
"""
 ANÁLISIS EXPLORATORIO DE DATOS FINANCIEROS

FASE 1: CARGA, LIMPIEZA Y EDA
"""

# 1. IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Librerías cargadas")

# 2. CARGAR DATOS
df = pd.read_csv('../data/dataset_final_para_modelos(m).csv', sep=';')
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df.set_index('date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

# 3. INGENIERÍA DE CARACTERÍSTICAS
df['daily_return'] = (df['close'] - df['open']) / df['open']
df['intraday_volatility'] = (df['high'] - df['low']) / df['close']
df['volume_scaled'] = df['volume'] / df['volume'].max()
df['price_range'] = df['high'] - df['low']

print("✅ Nuevas características: daily_return, intraday_volatility, volume_scaled, price_range")

# 4. DISTRIBUCIÓN DEL LABEL
plt.figure(figsize=(8, 5))
sns.countplot(x='Label', data=df, palette='viridis')
plt.title('Distribución de Tendencias (0=Baja, 1=Subida)')
plt.xlabel('Label')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('../outputs/graficos/distribucion_label.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

# 5. VOLUMEN POR TICKER
plt.figure(figsize=(12, 6))
sns.boxplot(x='Ticker', y='volume', data=df, palette='Set2')
plt.title('Volumen por Ticker')
plt.yscale('log')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/graficos/volumen_por_ticker.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

# 6. VOLATILIDAD VS RETORNO
plt.figure(figsize=(10, 6))
sns.scatterplot(x='intraday_volatility', y='daily_return', hue='Label', data=df, alpha=0.6)
plt.title('Volatilidad vs Retorno Diario')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/graficos/volatilidad_vs_retorno.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

# 7. PREPROCESAMIENTO DE TEXTO
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('spanish'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

df['text_clean'] = df['Headline_esp'].astype(str).apply(clean_text)

print("\n Ejemplo de texto procesado:")
print(f"Original: {df['Headline_esp'].iloc[0]}")
print(f"Procesado: {df['text_clean'].iloc[0]}")

# 8. MATRIZ DE CORRELACIÓN
plt.figure(figsize=(10, 8))
corr = df[['daily_return', 'intraday_volatility', 'volume_scaled', 'price_range', 'Label']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.savefig('../outputs/graficos/matriz_correlacion.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

# 9. GUARDAR DATASET PROCESADO
df.to_csv('../data/dataset_procesado.csv', sep=';')
print("✅ Dataset procesado guardado")

print("\n✅ ANÁLISIS EXPLORATORIO COMPLETADO")